<a href="https://colab.research.google.com/github/mariaaapetrovskaya/Python/blob/main/%D0%9F%D0%B5%D1%82%D1%80%D0%BE%D0%B2%D1%81%D0%BA%D0%B0%D1%8F_lr4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Лабораторная работа №4.

Как вы знаете, во время написания художественного текста авторы нередко испытывают муки выбора между тем или иным словом. Все эти правки остаются за кадром: текстологи изучают черновики и моделируют творческий процесс, анализируют замены и пытаются отследить изменения в авторском замысле.

Черновики читать сложно! Но всем однажды хочется заглянуть в закулисье творческого процесса. Давайте напишем текст, которого никогда не было, но который вполне мог бы существовать!

Задача: возьмите отрывок любого художественного текста (абзац или несколько строф). Замените все
- существительные,
- глаголы,
- прилагательные

на ближайшие ассоциаты той же части речи из векторной модели.


### Критерии оценивания:
1. Художественный текст токенизирован и лемматизирован. - 2 балла
2. Для знаменательных частей речи извлечены грамматические характеристики. - 2,5 балла
3. Для знаменательных частей речи подобран ближайший семантический ассоциат. - 2,5 балла
4. Замены всех частей речи стоят в правильной форме. - 3 балла

In [ ]:
# выберите нужные части речи


In [ ]:
# замените эти слова на ближайшие из модели


In [ ]:
# поставьте их в правильную форму

In [ ]:
# выведите альтернативную версию текста

In [2]:
!pip install razdel pymorphy3 gensim

In [3]:
import re
import pymorphy3

from razdel import tokenize, sentenize
from gensim.models import Word2Vec
from google.colab import files

In [4]:
morph = pymorphy3.MorphAnalyzer()

In [5]:
text = """Однажды утром, пробудившись от беспокойного сна, Грегор Замза увидел, что превратился в страшное насекомое. Он лежал на твердой, похожей на панцирь, спине, и, как только поднимал голову, мог видеть свой коричневый, выпуклый, разделенный чешуйками на полукружья живот, на котором еще кое-как держалось одеяло, готовое вот-вот сползти. Его многочисленные, невероятно тонкие по сравнению со всем телом ножки беспомощно мельтешили перед глазами."""

In [6]:
tokens = list(tokenize(text))

words = []

for token in tokens:
    word = token.text

    if re.fullmatch(r"[А-Яа-яЁё-]+", word):
        lemma = morph.parse(word.lower())[0].normal_form
        words.append(lemma)

print(words)

['однажды', 'утром', 'пробудиться', 'от', 'беспокойный', 'сон', 'грегор', 'замза', 'увидеть', 'что', 'превратиться', 'в', 'страшный', 'насекомое', 'он', 'лежать', 'на', 'твёрдый', 'похожий', 'на', 'панцирь', 'спина', 'и', 'как', 'только', 'поднимать', 'голова', 'мочь', 'видеть', 'свой', 'коричневый', 'выпуклый', 'разделить', 'чешуйка', 'на', 'полукружие', 'живот', 'на', 'который', 'ещё', 'кое-как', 'держаться', 'одеяло', 'готовый', 'вот-вот', 'сползти', 'он', 'многочисленный', 'невероятно', 'тонкий', 'по', 'сравнение', 'с', 'весь', 'тело', 'ножка', 'беспомощно', 'мельтешить', 'перед', 'глаз']


In [8]:
!pip -q install gensim pymorphy3 huggingface_hub

In [9]:
import re
import numpy as np
import scipy.linalg

if not hasattr(scipy.linalg, "triu"):
    scipy.linalg.triu = np.triu

from gensim.models import Word2Vec
from huggingface_hub import hf_hub_download
import pymorphy3

from collections import defaultdict

morph = pymorphy3.MorphAnalyzer()

In [19]:
import re
import pandas as pd
import pymorphy3

morph = pymorphy3.MorphAnalyzer()

POS_MAP = {
    "NOUN": "NOUN",
    "VERB": "VERB",
    "INFN": "VERB",
    "ADJF": "ADJF",
    "ADJS": "ADJF",
    "PRTF": "ADJF",
    "PRTS": "ADJF"
}

def get_pos(parsed):
    return POS_MAP.get(parsed.tag.POS, parsed.tag.POS)

In [20]:
original_text = """
Однажды утром, пробудившись от беспокойного сна, Грегор Замза увидел, что превратился в страшное насекомое. Он лежал на твердой, похожей на панцирь, спине, и, как только поднимал голову, мог видеть свой коричневый, выпуклый, разделенный чешуйками на полукружья живот, на котором еще кое-как держалось одеяло, готовое вот-вот сползти. Его многочисленные, невероятно тонкие по сравнению со всем телом ножки беспомощно мельтешили перед глазами.
"""

In [21]:
def find_associate(word):

    parsed = morph.parse(word.lower())[0]
    pos = get_pos(parsed)


    if pos not in {"NOUN", "VERB", "ADJF"}:
        return word, None, None


    if {"Name", "Surn", "Patr"}.intersection(parsed.tag.grammemes):
        return word, None, None

    model_word = parsed.normal_form + "_" + pos

    if model_word not in model.wv:
        return word, None, None

    candidates = model.wv.most_similar(model_word, topn=300)

    for candidate, similarity in candidates:

        if not candidate.endswith("_" + pos):
            continue

        lemma = candidate.rsplit("_", 1)[0]


        if not re.fullmatch(r"[а-яё]+", lemma):
            continue

        candidate_parse = morph.parse(lemma)[0]

        if get_pos(candidate_parse) != pos:
            continue

        if {"Name", "Surn", "Patr"}.intersection(candidate_parse.tag.grammemes):
            continue

        grammemes = set(parsed.tag.grammemes)

        if pos == "NOUN":
            grammemes &= {
                "nomn", "gent", "datv", "accs", "ablt", "loct",
                "sing", "plur"
            }

        elif pos == "ADJF":
            grammemes &= {
                "nomn", "gent", "datv", "accs", "ablt", "loct",
                "sing", "plur", "masc", "femn", "neut"
            }

        elif pos == "VERB":
            grammemes &= {
                "sing", "plur", "masc", "femn", "neut",
                "1per", "2per", "3per", "pres", "past", "futr",
                "indc", "impr"
            }

        inflected = candidate_parse.inflect(grammemes)

        if inflected:
            return inflected.word, lemma, similarity

    return word, None, None

In [22]:
replacements = []
unchanged = []

def replace_word(match):

    word = match.group(0)

    new_word, lemma, similarity = find_associate(word)

    if lemma is not None:

        if word[0].isupper():
            new_word = new_word.capitalize()

        replacements.append({
            "Исходное слово": word,
            "Замена": new_word,
            "Лемма замены": lemma,
            "Косинусн сходство": round(float(similarity), 3)
        })

    else:
        unchanged.append(word)

    return new_word


alternative_text = re.sub(
    r"[А-Яа-яЁё]+",
    replace_word,
    original_text
)

print("альт тектс:\n")
print(alternative_text)

альт тектс:


Однажды утром, пробудившись от судебного вида, Грегор Замза прочёл, что превратился в любимое лакей. Он оглянулся на готовой, крайней на панцирь, голосе, и, как только вскочил стол, понимал верить умоляющего голубой, пухлый, разделенный чешуйками на полукружья комик, на прошлом еще смешное-как ворчало тарантас, старое вот-вот ложиться. Его каждые, невероятно широкие по журналу со бледным грудью поручика беспомощно мельтешили перед руками.



In [23]:
df = pd.DataFrame(replacements)

display(df)

print("Колво замен:", len(replacements))
print("Колво слов без замены:", len(unchanged))

,Исходное слово,Замена,Лемма замены,Косинусн сходство
0,беспокойного,судебного,судебный,0.998
1,сна,вида,вид,0.999
2,увидел,прочёл,прочесть,0.999
3,страшное,любимое,любимый,0.999
4,насекомое,лакей,лакей,0.993
5,лежал,оглянулся,оглянуться,0.999
6,твердой,готовой,готовый,0.966
7,похожей,крайней,крайний,0.998
8,спине,голосе,голос,0.999
9,поднимал,вскочил,вскочить,0.998


Колво замен: 30
Колво слов без замены: 32


In [24]:
with open("/content/kafka_alternative.txt", "w", encoding="utf-8") as f:
    f.write(alternative_text)

df.to_csv(
    "/content/kafka_replacements.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Файлы сохранены!")

Файлы сохранены!
